## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-21-05-30-50 +0000,startribune,Friday rewind: Thrillers and blowouts mark Day...,https://www.startribune.com/live-scores-update...
1,2026-11-21-05-25-45 +0000,nyt,Federal Court Halts Pregnant Workers’ Protecti...,https://www.nytimes.com/2026/11/21/us/politics...
2,2026-11-21-05-15-41 +0000,bbc,Zelensky says Putin is a 'killer' after Trump ...,https://www.bbc.co.uk/news/articles/cxnv4zvmdy...
3,2026-11-21-05-13-03 +0000,nyt,"Alice Huang, Accused of Spying on Nvidia’s CEO...",https://www.nytimes.com/2026/11/20/us/politics...
4,2026-11-21-05-05-39 +0000,nypost,Heartbreaking development as search for missin...,https://nypost.com/2026/11/21/us-news/body-fou...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2466/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
17,trump,46
313,new,24
234,anthropic,20
56,he,15
463,tech,15


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
218,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...,135
141,2026-11-20-21-35-47 +0000,cbc,Trump threatens to penalize U.S. tech companie...,https://www.cbc.ca/news/world/ai-data-centres-...,116
127,2026-11-20-21-58-36 +0000,latimes,Trump says he'll penalize U.S. companies that ...,https://www.latimes.com/politics/story/2026-11...,105
294,2026-11-20-14-28-00 +0000,wsj,The Trump administration auctioned off parts o...,https://www.wsj.com/politics/the-u-s-is-auctio...,104
43,2026-11-21-01-03-00 +0000,wsj,During President Trump’s White House briefing ...,https://www.wsj.com/politics/elections/how-a-w...,99


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
218,135,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...
139,72,2026-11-20-21-43-22 +0000,bbc,What the new UK-EU deal could mean for future ...,https://www.bbc.co.uk/news/videos/c90k6gl3lyxe...
354,57,2026-11-20-10-08-25 +0000,wsj,"Stock Market News, Nov. 20, 2026: Tech Rout De...",https://www.wsj.com/livecoverage/stock-market-...
43,53,2026-11-21-01-03-00 +0000,wsj,During President Trump’s White House briefing ...,https://www.wsj.com/politics/elections/how-a-w...
170,48,2026-11-20-20-25-49 +0000,nypost,"‘GTA 6’ maker Take-Two rakes in nearly $4B, sh...",https://nypost.com/2026/11/20/business/take-tw...
91,48,2026-11-20-22-50-36 +0000,nypost,Russia open to ‘NATO-like’ status for Ukraine ...,https://nypost.com/2026/11/20/us-news/russia-o...
3,42,2026-11-21-05-13-03 +0000,nyt,"Alice Huang, Accused of Spying on Nvidia’s CEO...",https://www.nytimes.com/2026/11/20/us/politics...
208,40,2026-11-20-18-48-55 +0000,nypost,Senate approves bipartisan bill to force data ...,https://nypost.com/2026/11/20/us-news/senate-a...
370,39,2026-11-20-09-18-00 +0000,wsj,"British business bank OakNorth, whose sharehol...",https://www.wsj.com/economy/central-banking/oa...
341,35,2026-11-20-11-00-00 +0000,latimes,Arellano: Orange County was once California's ...,https://www.latimes.com/california/story/2026-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
